<a href="https://colab.research.google.com/github/patriciomontenegro/conectar_AI/blob/main/mecassist_IA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CREANDO UN AGENTE DE IA PARA LA EMPRESA MECASSIST

In [17]:
%%capture
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

In [18]:
!pip install langchain langchain-groq langchain-community sentence-transformers faiss-cpu pypdf -q
!pip install langchainhub -q

In [19]:
import os
from getpass import getpass
from langchain_groq import ChatGroq
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.tools import create_retriever_tool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
import langchainhub as hub


In [20]:
from google.colab import userdata
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

In [21]:
llm = ChatGroq(groq_api_key=GROQ_API_KEY, model="llama-3.3-70b-versatile", temperature=0.1)

In [22]:
rutas_politicas = {
                  "horarios": "/content/01_politica_horarios.pdf",
                  "servicio": "/content/02_politica_servicio.pdf",
                  "satisfaccion": "/content/03_politica_satisfaccion.pdf",
                  "garantia": "/content/04_politica_garantia.pdf"
                  }

In [23]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [24]:
recuperadores = {}

for politica, ruta in rutas_politicas.items():
    if os.path.exists(ruta):
        loader = PyPDFLoader(ruta)
        paginas = loader.load_and_split(text_splitter)
        vector_store = FAISS.from_documents(paginas, embeddings)
        # Guardamos el buscador de cada PDF
        recuperadores[politica] = vector_store.as_retriever(search_kwargs={"k": 2})
    else:
        print(f"Advertencia: No se encontró el archivo en {ruta}")

In [25]:
def obtener_contexto_relevante(consulta: str) -> str:
    """
    Analiza las palabras clave de la consulta del usuario para extraer el texto
    únicamente del PDF que corresponde a la pregunta.
    """
    consulta_lower = consulta.lower()
    contexto = ""

    # Identificación semántica simple y robusta
    if "horario" in consulta_lower or "hora" in consulta_lower or "atencion" in consulta_lower:
        if "horarios" in recuperadores:
            docs = recuperadores["horarios"].invoke(consulta)
            contexto += "\n[Documento: Política de Horarios]\n" + "\n".join([d.page_content for d in docs])

    if "servicio" in consulta_lower or "atender" in consulta_lower or "procedimiento" in consulta_lower:
        if "servicio" in recuperadores:
            docs = recuperadores["servicio"].invoke(consulta)
            contexto += "\n[Documento: Política de Servicio]\n" + "\n".join([d.page_content for d in docs])

    if "satisfaccion" in consulta_lower or "reclamo" in consulta_lower or "devolucion" in consulta_lower:
        if "satisfaccion" in recuperadores:
            docs = recuperadores["satisfaccion"].invoke(consulta)
            contexto += "\n[Documento: Política de Satisfacción]\n" + "\n".join([d.page_content for d in docs])

    if "garantia" in consulta_lower or "falla" in consulta_lower or "reparacion" in consulta_lower:
        if "garantia" in recuperadores:
            docs = recuperadores["garantia"].invoke(consulta)
            contexto += "\n[Documento: Política de Garantía]\n" + "\n".join([d.page_content for d in docs])

    return contexto if contexto != "" else "No se encontraron documentos específicos para esta consulta."

In [26]:
prompt_sistema = ChatPromptTemplate.from_messages([
    ("system",
     "Eres el asistente virtual de Mecassist, experto en atención al cliente. "
     "Tu única tarea es responder consultas de usuarios externos usando "
     "el fragmento de las políticas corporativas provisto abajo.\n\n"
     "REGLAS ESTRICTAS DE FORMATO Y ESTILO:\n"
     "1. Sé extremadamente directo, breve y conciso. Evita introducciones largas o saludos repetitivos.\n"
     "2. Responde en un máximo de 2 o 3 oraciones cortas (o un par de puntos clave si es una lista).\n"
     "3. Usa un tono amable pero corporativo.\n"
     "4. Responde basándote ÚNICAMENTE en el contexto proporcionado. No agregues datos extra ni suposiciones.\n"
     "5. Si la respuesta no está en el contexto, di estrictamente: 'Lo siento, no tengo registro de esa información en nuestras políticas actuales.'\n\n"
     "CONTEXTO EXTRAÍDO DE LOS PDF:\n{contexto}"),
    ("human", "{input}")
])

In [27]:
from langchain_core.output_parsers import StrOutputParser

In [28]:
cadena_mecassist = prompt_sistema | llm | StrOutputParser()

In [33]:
if __name__ == "__main__":
    print("\n--- Agente Mecassist Activo ---")

    pregunta_usuario = "¿las reparaciones tienen garantía?"

    contexto_extraido = obtener_contexto_relevante(pregunta_usuario)

    respuesta_final = cadena_mecassist.invoke({
        "contexto": contexto_extraido,
        "input": pregunta_usuario
    })


--- Agente Mecassist Activo ---


In [34]:
print(f"\nCliente: {pregunta_usuario}")
print(f"\nAgente Mecassist:\n{respuesta_final}")


Cliente: ¿las reparaciones tienen garantía?

Agente Mecassist:
Sí, las reparaciones realizadas por nuestro taller cuentan con garantía. Los componentes instalados tienen la garantía legal y de fábrica correspondiente, gestionada directamente por nosotros ante el proveedor.
